In [ ]:
!pip install -q langchain-openai langchain-core requests


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 1.	Tool Creation
# 2.	Tool Binding
# 3.	Tool Calling
# 4.	Tool Execution
# 5.	Context aggregation (Human → AI → Tool messages)
# 6.	Final LLM response

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

c:\Users\smi68\Desktop\My_Learning\Artificial-Intelligence\LangChain\LangChain_Code\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# tool create
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [5]:
#Check if the tool is properly created
print(multiply.invoke({'a':3, 'b':4}))

12


In [6]:
#Tool having mainly three properties - name, description and args
multiply.name
multiply.description
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [7]:
# tool binding

#Create the instance of openai model
import os
from dotenv import load_dotenv      

load_dotenv("C:/Users/smi68/Desktop/My_Learning/Artificial-Intelligence/LangChain/LangChain_Code/.env")

# Create a connection to a Hugging Face hosted model (cloud-based)

llm = ChatOpenAI(
    model='mistralai/mistral-small-2506',
    api_key=os.getenv("LLM_API_KEY"),     # API key from .env
    base_url=os.getenv("LLM_BASE_URL")    # Custom provider endpoint
)

#Check if the  llm is working properly
llm.invoke("Hi")

#bind llm with the tools using bind_tools and pass the tool to that function
llm_with_tools = llm.bind_tools([multiply])

#Invoke the tool with the prompt - It is the normal message which we are giving as prompt and we get the normal reply
llm_with_tools.invoke('Hi how are you')

AuthenticationError: Error code: 401 - {'error': 'Unauthorized, Token expired', 'details': 'Signature has expired'}

In [ ]:
#Next is the prompt for which it is necessary to call the tool
#As the prompt is asked by user so we convert it into HumanMessage
query = HumanMessage('can you multiply 3 with 1000')

#We create the messages list to which we are adding the HumanMessage
messages = [query]

#We invokecalling the llm with the human message
#We get the result - but there will nothing in the content
#From the below call we get the information which tool to call and the tool is called by the developer later - This is tool calling result
result = llm_with_tools.invoke(messages)

#we will append the ai message into messages
messages.append(reslut)

In [ ]:
#Execution of the tool - where we pass the tool to the function method 
#It will exceute the function and give the result
tool_result = multiply.invoke(result.tool_calls[0])

In [ ]:
#When we print it we the get the tool message as a result
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_RxxH1pPDylDECUwpRe7MXkJi')

In [ ]:
#We will append the tool message into the messages list
messages.append(tool_result)

In [ ]:
#when we print the messages it have everything - human message, ai message and tool messagae
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_RxxH1pPDylDECUwpRe7MXkJi', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 63, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BR8rS3DNc8cckcVLJMmBDxHENKUlV', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-8035ac83-7820-4681-b8c0-1d15aa24ca77-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_RxxH1pPDylDECUwpRe7MXkJi', 'type': 'tool_call'}], usage_metadata={'input_tokens': 63, 'output_tokens': 19

In [ ]:
#we invoke the llmWithTools with the messages we got
#llm will be having the complete information - human message, ai message and tool message - using this conversation it will generate the answer
lm_with_tools.invoke(messages).content

'The product of 3 and 1000 is 3000.'